# Test / Evaluation — xuất dữ liệu cho báo cáo

Load một checkpoint đã train (KHÔNG train lại) rồi sinh **dữ liệu cho report**:
- Bảng metrics đầy đủ (Acc / Precision / Recall / F1 / IoU).
- **Threshold analysis** — quét ngưỡng quyết định, tìm best-F1 (như báo cáo mẫu).
- Phân phối IoU theo từng mẫu.
- Hình **qualitative** (explode + tô đỏ fracture), lưu PNG vào `/kaggle/working/test_figs/`.

**Add Input:**
- Dataset chứa **checkpoint** (`.pt`).
- Nếu `DATASET='fb'`: thêm `Fantastic_Breaks_v1`.
- Nếu `DATASET='bb'`: thêm dataset Breaking Bad đã decompress.

## 1. Setup

In [ ]:
import os, sys, random, json, time
from pathlib import Path
from collections import deque, defaultdict

!pip install -q trimesh potpourri3d robust_laplacian plyfile scikit-learn fast-simplification kaleido

DIFFNET_DIR = '/kaggle/working/diffusion-net'
if not os.path.exists(DIFFNET_DIR):
    !git clone https://github.com/nmwsharp/diffusion-net.git {DIFFNET_DIR}
sys.path.append(f'{DIFFNET_DIR}/src')

import numpy as np
import torch
import torch.nn.functional as F
import trimesh
from scipy.spatial import cKDTree
import diffusion_net
try:
    import fast_simplification
except Exception:
    fast_simplification = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 2. Cấu hình — đổi DATASET và kiểm tra kiến trúc khớp checkpoint

In [ ]:
DATASET = 'fb'             # 'fb' (Fantastic Breaks test) | 'bb' (Breaking Bad val)

# --- Kiến trúc: PHẢI khớp checkpoint (model hiện tại C_WIDTH=128) ---
INPUT_FEATURES = 'xyz_hks_curv'
N_HKS   = 16
K_EIG   = 128
C_WIDTH = 128
N_BLOCK = 4
C_IN = (3 if 'xyz' in INPUT_FEATURES else 0) + (N_HKS if 'hks' in INPUT_FEATURES else 0) + (1 if 'curv' in INPUT_FEATURES else 0)

# --- FB ---
DECIMATE_VERTS = 22000
DILATE_RINGS   = 1
N_TEST_FB      = 50        # tái tạo CÙNG split test như fb_transfer (SEED=42)
N_VAL_FB       = 20
# --- BB ---
LABEL_THRESHOLD_RATIO = 0.005

# --- Test/visualize ---
N_QUALITATIVE  = 4         # số mẫu vẽ hình
EXPLODE        = 0.3
THRESHOLDS     = [round(t, 2) for t in np.arange(0.05, 0.96, 0.05)]

OP_CACHE  = '/kaggle/temp/test_op_cache'
FB_CACHE  = '/kaggle/temp/test_fb_cache'
print(f'DATASET={DATASET}  C_in={C_IN}  C_width={C_WIDTH}')

## 3. Dò checkpoint + dataset

In [ ]:
INPUT = Path('/kaggle/input')
print('Datasets attached:')
for d in INPUT.iterdir():
    print('  -', d.name)

pt_files = list(INPUT.rglob('*.pt'))
assert pt_files, '❌ Không thấy checkpoint .pt — attach dataset chứa file model.'
CKPT = str(pt_files[0])
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
print(f'\n✅ Checkpoint: {CKPT}')
if 'config' in ck:
    print('   config:', ck['config'])
print('   best_val_iou:', ck.get('best_val_iou', ck.get('val_best_iou', 'N/A')))

if DATASET == 'fb':
    meta_files = list(INPUT.rglob('meta_*.npz'))
    sample_dirs = sorted(set(m.parent for m in meta_files))
    assert sample_dirs, '❌ DATASET=fb nhưng không thấy meta_*.npz (Fantastic_Breaks_v1).'
    print(f'✅ Fantastic Breaks: {len(sample_dirs)} mẫu')
else:
    def find_root(base, md_=5):
        best, bc = None, 0; q = deque([(base, 0)])
        while q:
            p, d = q.popleft()
            if d > md_: continue
            try: ch = list(p.iterdir())
            except OSError: continue
            n = sum(1 for c in ch if c.is_dir() and c.name.endswith('_sf'))
            if n > bc: bc, best = n, p
            for c in ch:
                if c.is_dir() and not c.name.endswith('_sf'): q.append((c, d+1))
        return best
    def find_file(base, name, md_=6):
        q = deque([(base, 0)])
        while q:
            p, d = q.popleft()
            if d > md_: continue
            try: ch = list(p.iterdir())
            except OSError: continue
            for c in ch:
                if c.is_file() and c.name == name: return c
                if c.is_dir(): q.append((c, d+1))
        return None
    ARTIFACT_ROOT = find_root(INPUT); val_txt = find_file(INPUT, 'artifact.val.txt')
    assert ARTIFACT_ROOT and val_txt, '❌ DATASET=bb nhưng không thấy artifact data / val.txt.'
    with open(val_txt) as fp:
        val_ids = [l.strip().split('/')[-1] for l in fp if l.strip()]
    print(f'✅ Breaking Bad val: {len(val_ids)} objects')

## 4. Dataset + features (khớp pipeline đã train)

In [ ]:
from torch.utils.data import Dataset

# ---------- BB dataset (proximity labels) ----------
class BBFractureDataset(Dataset):
    def __init__(self, data_root, object_ids, op_cache_dir, mesh_cache_dir,
                 label_threshold_ratio=0.005, max_vertices=30000, k_eig=128):
        self.data_root = Path(data_root)
        self.op = Path(op_cache_dir); self.op.mkdir(parents=True, exist_ok=True)
        self.mc = Path(mesh_cache_dir); self.mc.mkdir(parents=True, exist_ok=True)
        self.thr = label_threshold_ratio; self.maxv = max_vertices; self.k = k_eig
        self.samples = []
        for oid in object_ids:
            od = self.data_root / oid
            if not od.exists(): continue
            for fr in sorted(od.iterdir()):
                if fr.is_dir() and any(fr.glob('piece_*.obj')):
                    self.samples.append((oid, fr.name))
        print(f'  BB val dataset: {len(self.samples)} fractures')
    def __len__(self): return len(self.samples)
    def _raw(self, fd):
        pcs = []
        for pf in sorted(fd.glob('piece_*.obj')):
            m = trimesh.load(str(pf), process=False)
            pcs.append((np.asarray(m.vertices, np.float32), np.asarray(m.faces, np.int64)))
        if not pcs: return None
        allv = np.concatenate([v for v, _ in pcs])
        thr = np.linalg.norm(allv.max(0)-allv.min(0)) * self.thr
        mv, mf, ml, off = [], [], [], 0
        for i, (v, f) in enumerate(pcs):
            oth = [pv for j, (pv, _) in enumerate(pcs) if j != i]
            lab = (cKDTree(np.concatenate(oth)).query(v, k=1)[0] < thr).astype(np.int64) if oth else np.zeros(len(v), np.int64)
            mv.append(v); mf.append(f+off); ml.append(lab); off += len(v)
        verts = np.concatenate(mv); faces = np.concatenate(mf).astype(np.int64); labels = np.concatenate(ml)
        verts = verts - verts.mean(0); sc = np.linalg.norm(verts.max(0)-verts.min(0))
        if sc > 0: verts = verts/sc
        return verts.astype(np.float32), faces, labels
    def _get(self, oid, fn):
        cf = self.mc / f'{oid}__{fn}__t{self.thr}.npz'
        if cf.exists():
            d = np.load(cf); return d['verts'], d['faces'], d['labels']
        r = self._raw(self.data_root/oid/fn)
        if r is None: return None
        np.savez(cf, verts=r[0], faces=r[1], labels=r[2]); return r
    def __getitem__(self, idx):
        oid, fn = self.samples[idx]; r = self._get(oid, fn)
        if r is None: return None
        vn, fn2, ln = r
        if len(vn) > self.maxv: return None
        verts = torch.tensor(vn); faces = torch.tensor(fn2, dtype=torch.long); labels = torch.tensor(ln, dtype=torch.long)
        try:
            _, mass, L, evals, evecs, gX, gY = diffusion_net.geometry.get_operators(
                verts, faces, k_eig=self.k, op_cache_dir=str(self.op))
        except Exception as e:
            print('op fail', oid, fn, e); return None
        return {'verts': verts, 'faces': faces, 'labels': labels, 'mass': mass, 'L': L,
                'evals': evals, 'evecs': evecs, 'gradX': gX, 'gradY': gY, 'obj_id': oid, 'frac_name': fn}

# ---------- FB preprocessing + dataset ----------
def preprocess_fb(sd, target_verts, dilate_rings):
    sd = Path(sd)
    mb = sorted(sd.glob('model_b_*.ply')); mt = sorted(sd.glob('meta_*.npz'))
    if not mb or not mt: return None
    m = trimesh.load(str(mb[0]), process=False)
    V0 = np.ascontiguousarray(np.asarray(m.vertices, np.float64))
    F0 = np.ascontiguousarray(np.asarray(m.faces, np.int32))
    mask = np.load(str(mt[0]))['mask'].astype(bool)
    if len(V0) != len(mask) or len(F0) < 10: return None
    red = max(0.05, min(0.99, 1.0 - (target_verts*2)/len(F0)))
    Vd, Fd = fast_simplification.simplify(V0, F0, target_reduction=red)
    Vd = np.asarray(Vd, np.float32); Fd = np.asarray(Fd, np.int64)
    if len(Vd) < 100 or len(Fd) < 100: return None
    labels = mask[cKDTree(V0).query(Vd, k=1)[1]].astype(np.int64)
    if dilate_rings > 0:
        adj = defaultdict(set)
        for a, b, c in Fd:
            adj[a].update((b, c)); adj[b].update((a, c)); adj[c].update((a, b))
        for _ in range(dilate_rings):
            grow = set()
            for v in np.where(labels == 1)[0]: grow.update(adj[int(v)])
            if grow: labels[list(grow)] = 1
    Vd = Vd - Vd.mean(0); sc = np.linalg.norm(Vd.max(0)-Vd.min(0))
    if sc > 0: Vd = Vd/sc
    return Vd.astype(np.float32), Fd, labels

class FBDataset(Dataset):
    def __init__(self, files, op_cache_dir, k_eig):
        self.files = list(files); self.op = Path(op_cache_dir); self.op.mkdir(parents=True, exist_ok=True); self.k = k_eig
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        d = np.load(self.files[i])
        verts = torch.tensor(d['verts'], dtype=torch.float32)
        faces = torch.tensor(d['faces'], dtype=torch.long)
        labels = torch.tensor(d['labels'], dtype=torch.long)
        try:
            _, mass, L, evals, evecs, gX, gY = diffusion_net.geometry.get_operators(
                verts, faces, k_eig=self.k, op_cache_dir=str(self.op))
        except Exception as e:
            print('op fail', self.files[i].name, e); return None
        return {'verts': verts, 'faces': faces, 'labels': labels, 'mass': mass, 'L': L,
                'evals': evals, 'evecs': evecs, 'gradX': gX, 'gradY': gY}

def compute_metrics(preds, labels):
    preds = preds.flatten(); labels = labels.flatten()
    tp = ((preds==1)&(labels==1)).sum().item(); fp = ((preds==1)&(labels==0)).sum().item()
    fn = ((preds==0)&(labels==1)).sum().item(); tn = ((preds==0)&(labels==0)).sum().item()
    tot = tp+fp+fn+tn
    return {'acc': (tp+tn)/tot if tot else 0,
            'precision': tp/(tp+fp) if (tp+fp) else 0,
            'recall': tp/(tp+fn) if (tp+fn) else 0,
            'f1': (2*tp)/(2*tp+fp+fn) if (2*tp+fp+fn) else 0,
            'iou': tp/(tp+fp+fn) if (tp+fp+fn) else 0}

def _std_log(x):
    x = torch.log(x.clamp_min(1e-8)); return (x - x.mean(0, keepdim=True))/(x.std(0, keepdim=True)+1e-6)

def get_features(sample):
    verts = sample['verts'].to(device); feats = []
    if 'xyz' in INPUT_FEATURES: feats.append(verts)
    if 'hks' in INPUT_FEATURES:
        hks = diffusion_net.geometry.compute_hks_autoscale(sample['evals'].to(device), sample['evecs'].to(device), N_HKS)
        feats.append(_std_log(hks))
    if 'curv' in INPUT_FEATURES:
        L = sample['L'].to(device); mass = sample['mass'].to(device)
        Hv = torch.sparse.mm(L, verts)/mass.clamp_min(1e-8).unsqueeze(-1)
        feats.append(_std_log(Hv.norm(dim=-1, keepdim=True)))
    return feats[0] if len(feats) == 1 else torch.cat(feats, dim=-1)

def forward_sample(model, sample):
    return model(x_in=get_features(sample), mass=sample['mass'].to(device), L=sample['L'].to(device),
        evals=sample['evals'].to(device), evecs=sample['evecs'].to(device),
        gradX=sample['gradX'].to(device), gradY=sample['gradY'].to(device), faces=sample['faces'].to(device))

## 5. Dựng test dataset + load model

In [ ]:
if DATASET == 'fb':
    Path(FB_CACHE).mkdir(parents=True, exist_ok=True)
    cache_files = []
    for i, sd in enumerate(sample_dirs):
        out = Path(FB_CACHE) / f'fb_{i:04d}.npz'
        if not out.exists():
            r = preprocess_fb(sd, DECIMATE_VERTS, DILATE_RINGS)
            if r is None: continue
            np.savez(out, verts=r[0], faces=r[1], labels=r[2])
        cache_files.append(out)
        if (i+1) % 25 == 0: print(f'  preprocess {i+1}/{len(sample_dirs)}')
    rng = random.Random(SEED); files = list(cache_files); rng.shuffle(files)
    test_files = files[:N_TEST_FB]          # CÙNG 50 mẫu test như fb_transfer
    test_ds = FBDataset(test_files, OP_CACHE, K_EIG)
    print(f'FB test: {len(test_ds)} mẫu')
else:
    test_ds = BBFractureDataset(str(ARTIFACT_ROOT), val_ids, OP_CACHE,
                                '/kaggle/temp/test_bb_cache', LABEL_THRESHOLD_RATIO, k_eig=K_EIG)

model = diffusion_net.layers.DiffusionNet(C_in=C_IN, C_out=2, C_width=C_WIDTH,
        N_block=N_BLOCK, outputs_at='vertices', dropout=True).to(device)
model.load_state_dict(ck['model_state_dict'])
model.eval()
print(f'✅ Model loaded: {sum(p.numel() for p in model.parameters()):,} params')

## 6. Đánh giá đầy đủ (argmax) + thu thập xác suất

In [ ]:
@torch.no_grad()
def collect(model, dataset):
    model.eval(); P, Y, per = [], [], []
    for j in range(len(dataset)):
        s = dataset[j]
        if s is None: continue
        prob = torch.softmax(forward_sample(model, s), dim=-1)[:, 1].cpu().numpy()
        y = s['labels'].numpy()
        P.append(prob); Y.append(y)
        pr = (prob >= 0.5).astype(int)
        per.append(compute_metrics(torch.tensor(pr), torch.tensor(y))['iou'])
    return np.concatenate(P), np.concatenate(Y), np.array(per)

probs, ys, per_iou = collect(model, test_ds)
overall = compute_metrics(torch.tensor((probs >= 0.5).astype(int)), torch.tensor(ys))
print('=== OVERALL (threshold 0.5, argmax) ===')
for k, v in overall.items(): print(f'  {k:10s}: {v:.4f}')

## 7. Threshold analysis — quét ngưỡng, tìm best-F1 (như báo cáo mẫu)

In [ ]:
def m_at(prob, y, t):
    pred = (prob >= t).astype(int)
    tp = ((pred==1)&(y==1)).sum(); fp = ((pred==1)&(y==0)).sum(); fn = ((pred==0)&(y==1)).sum()
    prec = tp/(tp+fp) if (tp+fp) else 0; rec = tp/(tp+fn) if (tp+fn) else 0
    f1 = (2*tp)/(2*tp+fp+fn) if (2*tp+fp+fn) else 0; iou = tp/(tp+fp+fn) if (tp+fp+fn) else 0
    return prec, rec, f1, iou

print(f'{"thresh":>8}{"Prec":>9}{"Recall":>9}{"F1":>9}{"IoU":>9}')
rows = []
for t in THRESHOLDS:
    p, r, f1, iou = m_at(probs, ys, t)
    rows.append((t, p, r, f1, iou))
    print(f'{t:>8.2f}{p:>9.3f}{r:>9.3f}{f1:>9.3f}{iou:>9.3f}')

best_f1 = max(rows, key=lambda x: x[3])
best_iou = max(rows, key=lambda x: x[4])
print(f'\n► Best F1  @ threshold={best_f1[0]:.2f}: F1={best_f1[3]:.4f} IoU={best_f1[4]:.4f}')
print(f'► Best IoU @ threshold={best_iou[0]:.2f}: IoU={best_iou[4]:.4f} F1={best_iou[3]:.4f}')
print(f'  (so với threshold 0.5: F1={overall["f1"]:.4f} IoU={overall["iou"]:.4f})')

## 8. Phân phối IoU theo từng mẫu

In [ ]:
import matplotlib.pyplot as plt
print(f'Per-sample IoU (n={len(per_iou)}): mean={per_iou.mean():.3f}  '
      f'median={np.median(per_iou):.3f}  min={per_iou.min():.3f}  max={per_iou.max():.3f}')
plt.figure(figsize=(7, 3.5))
plt.hist(per_iou, bins=15, color='steelblue', edgecolor='white')
plt.axvline(per_iou.mean(), color='crimson', ls='--', label=f'mean={per_iou.mean():.3f}')
plt.xlabel('Per-sample fracture IoU'); plt.ylabel('Count'); plt.legend(); plt.tight_layout()
os.makedirs('/kaggle/working/test_figs', exist_ok=True)
plt.savefig('/kaggle/working/test_figs/iou_distribution.png', dpi=120, bbox_inches='tight'); plt.show()

## 9. Hình qualitative: explode + tô đỏ fracture (lưu PNG cho báo cáo)

In [ ]:
import plotly.graph_objects as go

def _components(faces, n):
    parent = np.arange(n)
    def find(x):
        root = x
        while parent[root] != root: root = parent[root]
        while parent[x] != root: parent[x], x = root, parent[x]
        return root
    for a, b, c in faces:
        for u, w in ((int(a), int(b)), (int(b), int(c))):
            ru, rw = find(u), find(w)
            if ru != rw: parent[ru] = rw
    return np.array([find(i) for i in range(n)])

@torch.no_grad()
def make_fig(sample, model, explode=0.3, show='pred'):
    model.eval()
    pred = forward_sample(model, sample).argmax(-1).cpu().numpy(); gt = sample['labels'].numpy()
    v = sample['verts'].numpy().copy(); f = sample['faces'].numpy()
    comp = _components(f, len(v)); ids = np.unique(comp); gc = v.mean(0)
    if len(ids) > 1:
        for cid in ids:
            mk = comp == cid; d = v[mk].mean(0) - gc; nrm = np.linalg.norm(d)
            if nrm > 1e-6: v[mk] += (d/nrm)*explode
    color = (pred if show == 'pred' else gt).astype(np.float32)
    m = compute_metrics(torch.tensor(pred), torch.tensor(gt))
    fig = go.Figure(go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2], i=f[:,0], j=f[:,1], k=f[:,2],
        intensity=color, colorscale=[[0,'lightgray'],[1,'crimson']], cmin=0, cmax=1,
        flatshading=True, showscale=False))
    fig.update_layout(width=820, height=620, scene=dict(aspectmode='data'),
        title=f'{"Prediction" if show=="pred" else "Ground Truth"} (đỏ=fracture) — '
              f'{len(ids)} mảnh, IoU={m["iou"]:.3f}')
    return fig

os.makedirs('/kaggle/working/test_figs', exist_ok=True)
for idx in range(N_QUALITATIVE):
    s = test_ds[idx]
    if s is None: continue
    for sh in ['gt', 'pred']:
        fig = make_fig(s, model, EXPLODE, show=sh)
        try:
            fig.write_image(f'/kaggle/working/test_figs/sample{idx}_{sh}.png', scale=2)
        except Exception as e:
            print('⚠️ write_image lỗi (kaleido?), hãy chụp màn hình thay thế:', e)
        fig.show()
print('✅ Hình lưu ở /kaggle/working/test_figs/')

## 10. Lưu tổng hợp metrics cho báo cáo

In [ ]:
summary = {
    'dataset': DATASET, 'checkpoint': os.path.basename(CKPT),
    'overall_thr0.5': overall,
    'threshold_sweep': [{'t': t, 'precision': p, 'recall': r, 'f1': f1, 'iou': iou} for t, p, r, f1, iou in rows],
    'best_f1': {'threshold': best_f1[0], 'f1': best_f1[3], 'iou': best_f1[4]},
    'best_iou': {'threshold': best_iou[0], 'iou': best_iou[4], 'f1': best_iou[3]},
    'per_sample_iou': {'mean': float(per_iou.mean()), 'median': float(np.median(per_iou)),
                       'min': float(per_iou.min()), 'max': float(per_iou.max()), 'n': int(len(per_iou))},
}
os.makedirs('/kaggle/working/test_figs', exist_ok=True)
with open('/kaggle/working/test_figs/test_summary.json', 'w') as fp:
    json.dump(summary, fp, indent=2)
print('✅ Saved /kaggle/working/test_figs/test_summary.json')
!ls -lah /kaggle/working/test_figs/
from IPython.display import FileLink
FileLink('test_figs/test_summary.json')